# EMT Ph3 VSI Voltage-Control VCO with Power-Flow Initialization

This notebook is a Python/Jupyter version of the C++ example `EMT_Slack_PiLine_VSI_VoltageControlled_LoadStep_with_PF_Init`.

It simulates the implemented GFM reacting to load steps inside the grid.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import dpsimpy

emt = dpsimpy.emt
sp = dpsimpy.sp

ph3 = emt.ph3
sp_ph1 = sp.ph1

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
PhaseType = dpsimpy.PhaseType
PowerflowBusType = dpsimpy.PowerflowBusType
Solver = dpsimpy.Solver
SwitchEvent3Ph = dpsimpy.event.SwitchEvent3Ph

# Simulation Parameters

The simulation parameters are derived from "Yazdani, A., & Iravani, R. (2010). Voltage-sourced converters in power systems: modeling, control, and applications. John Wiley & Sons." (Chapter 9.3)

In [ ]:
# Simulation setup
final_time = 1.0
time_step = 100e-6
system_frequency = 60.0
omega_null = 2.0 * np.pi * system_frequency

sim_name = "EMT_VSI_GFM"
sim_name_pf = sim_name + "_PF"
sim_name_emt = sim_name + "_EMT"

pv_with_control = True

# Low-voltage base
v_base = 400.0

# Line / feeder (Yazdani example)
length = 5.0
line_r = 0.5 * length
line_l = (0.5 / 314.0) * length
line_c = (50e-6 / 314.0) * length
line_g = 0.0

# Load 1 (Yazdani)
load1_r = 83e-3
load1_l = 137e-6

# Load 2 (Yazdani)
yazdani_res2 = 50e-3
yazdani_ind2 = 68e-6
yazdani_cap2 = 13.55e-3

# VSI voltage references
vd_ref = 400.0
vq_ref = 0.0

# Voltage controller
kp_voltage_ctrl = 1.6725
ki_voltage_ctrl = 374.64

# Current controller
kp_curr_ctrl = 0.2
ki_curr_ctrl = 4.14

# VSI filter parameters
lf = 100e-6
cf = 2.5e-3
rf = 2.07e-3
rc = 1e-5
tau = 0.5e-3

# Initial controller states
phi_d_init = 0.0
phi_q_init = 0.0
gamma_d_init = 0.0
gamma_q_init = 0.0
theta_pll_init = 0.0
phi_pll_init = 0.0

# PLL (VCO mode)
kp_pll = 0.0
ki_pll = 0.0
omega_cutoff = omega_null

# Switch/fault timing
switch_open = 1e9
switch_closed = 1e-9

start_time_fault1 = 0.2
end_time_fault1 = 0.8
start_time_fault2 = 0.4
end_time_fault2 = 0.6

## Helper functions

In [ ]:
def three_phase_parameter(value):
    return dpsimpy.Math.single_phase_parameter_to_three_phase(value)


def three_phase_power(value):
    return dpsimpy.Math.single_phase_power_to_three_phase(value)


def log_attr(logger, name, obj, attr_name):
    if hasattr(obj, "attr"):
        logger.log_attribute(name, obj.attr(attr_name))
    else:
        logger.log_attribute(name, obj.attribute(attr_name))


def load_csv(sim_name):
    path = Path("logs") / sim_name / f"{sim_name}.csv"
    return pd.read_csv(path, skipinitialspace=True)


def signal_columns(df, prefix):
    return [c for c in df.columns if c == prefix or c.startswith(prefix + "_")]

## Compute load powers

The C++ example computes single-phase complex powers and then converts them to three-phase powers for the EMT `RXLoad` components.

In [ ]:
load1_s = 3.0 * v_base**2 / complex(load1_r, load1_l * omega_null)
load1_p = load1_s.real
load1_q = load1_s.imag

load2_impedance = complex(
    yazdani_res2,
    yazdani_ind2 * omega_null - 1.0 / (omega_null * yazdani_cap2),
)
load2_s = 3.0 * v_base**2 / load2_impedance
load2_p = load2_s.real
load2_q = load2_s.imag

print(f"Load 1: P={load1_p:.6g} W, Q={load1_q:.6g} var")
print(f"Load 2: P={load2_p:.6g} W, Q={load2_q:.6g} var")

## SP-domain power-flow initialization

In [ ]:
def build_and_run_powerflow():
    Logger.set_log_dir("logs/" + sim_name_pf)

    time_step_pf = final_time
    final_time_pf = final_time + time_step_pf

    n1_pf = sp.SimNode("n1", PhaseType.Single)
    n2_pf = sp.SimNode("n2", PhaseType.Single)

    extnet_pf = sp_ph1.NetworkInjection("Slack")
    extnet_pf.set_parameters(v_base)
    extnet_pf.set_base_voltage(v_base)
    extnet_pf.modify_power_flow_bus_type(PowerflowBusType.VD)

    line_pf = sp_ph1.PiLine("PiLine")
    line_pf.set_parameters(line_r, line_l, line_c)
    line_pf.set_base_voltage(v_base)

    extnet_pf.connect([n1_pf])
    line_pf.connect([n1_pf, n2_pf])

    system_pf = SystemTopology(
        system_frequency,
        [n1_pf, n2_pf],
        [line_pf, extnet_pf],
    )

    logger_pf = Logger(sim_name_pf)
    log_attr(logger_pf, "v1", n1_pf, "v")
    log_attr(logger_pf, "v2", n2_pf, "v")

    sim_pf = Simulation(sim_name_pf)
    sim_pf.set_system(system_pf)
    sim_pf.set_time_step(time_step_pf)
    sim_pf.set_final_time(final_time_pf)
    sim_pf.set_domain(Domain.SP)
    sim_pf.set_solver(Solver.NRP)
    sim_pf.set_solver_component_behaviour(dpsimpy.SolverBehaviour.Initialization)
    sim_pf.do_init_from_nodes_and_terminals(False)
    sim_pf.add_logger(logger_pf)
    sim_pf.run()

    return system_pf


system_pf = build_and_run_powerflow()

## EMT-domain dynamic simulation

In [ ]:
def build_and_run_emt(system_pf):
    Logger.set_log_dir("logs/" + sim_name_emt)

    time_step_emt = time_step
    final_time_emt = final_time + time_step_emt

    n1_emt = emt.SimNode("n1", PhaseType.ABC)
    n2_emt = emt.SimNode("n2", PhaseType.ABC)
    n3_emt = emt.SimNode("n3", PhaseType.ABC)
    n4_emt = emt.SimNode("n4", PhaseType.ABC)

    load_emt_1 = ph3.RXLoad("Load1")
    load_emt_1.set_parameters(
        three_phase_power(load1_p),
        three_phase_power(load1_q),
        v_base,
    )

    load_emt_2 = ph3.RXLoad("Load2")
    load_emt_2.set_parameters(
        three_phase_power(load2_p),
        three_phase_power(load2_q),
        v_base,
    )

    res_on_emt = ph3.Resistor("ResOn")
    res_on_emt.set_parameters(three_phase_parameter(line_r))

    pv = ph3.VSIVoltageControlVCO("pv")
    pv.set_parameters(omega_null, vd_ref, vq_ref)
    pv.set_controller_parameters(
        kp_voltage_ctrl,
        ki_voltage_ctrl,
        kp_curr_ctrl,
        ki_curr_ctrl,
        omega_null,
    )
    pv.set_filter_parameters(lf, cf, rf, rc)
    pv.set_initial_state_values(phi_d_init, phi_q_init, gamma_d_init, gamma_q_init)
    pv.with_control(pv_with_control)

    fault1_emt = ph3.Switch("Switch1")
    fault1_emt.set_parameters(
        three_phase_parameter(switch_open),
        three_phase_parameter(switch_closed),
    )
    fault1_emt.open()

    fault2_emt = ph3.Switch("Switch2")
    fault2_emt.set_parameters(
        three_phase_parameter(switch_open),
        three_phase_parameter(switch_closed),
    )
    fault2_emt.open()

    pv.connect([n1_emt])
    res_on_emt.connect([n1_emt, n2_emt])
    fault1_emt.connect([n2_emt, n3_emt])
    load_emt_1.connect([n3_emt])
    fault2_emt.connect([n3_emt, n4_emt])
    load_emt_2.connect([n4_emt])

    system_emt = SystemTopology(
        system_frequency,
        [n1_emt, n2_emt, n3_emt, n4_emt],
        [res_on_emt, fault1_emt, fault2_emt, load_emt_1, load_emt_2, pv],
    )

    system_emt.init_with_powerflow(system_pf, Domain.EMT)

    logger_emt = Logger(sim_name_emt)
    log_attr(logger_emt, "Voltage_Node_1_PCC", n1_emt, "v")
    log_attr(logger_emt, "Voltage_Node_2", n2_emt, "v")
    log_attr(logger_emt, "Voltage_Node_3", n3_emt, "v")
    log_attr(logger_emt, "Voltage_Source", pv, "Vs")
    log_attr(logger_emt, "Current_RLC", pv, "i_intf")
    log_attr(logger_emt, "VCO_Phase", pv, "VCO_output")
    log_attr(logger_emt, "P_elec", pv, "P_elec")
    log_attr(logger_emt, "Q_elec", pv, "Q_elec")

    sim = Simulation(sim_name_emt)

    sim.add_event(SwitchEvent3Ph(start_time_fault1, fault1_emt, True))
    sim.add_event(SwitchEvent3Ph(end_time_fault1, fault1_emt, False))
    sim.add_event(SwitchEvent3Ph(start_time_fault2, fault2_emt, True))
    sim.add_event(SwitchEvent3Ph(end_time_fault2, fault2_emt, False))

    sim.set_system(system_emt)
    sim.set_time_step(time_step_emt)
    sim.set_final_time(final_time_emt)
    sim.set_domain(Domain.EMT)
    sim.add_logger(logger_emt)
    sim.run()

    return system_emt


system_emt = build_and_run_emt(system_pf)

## Read and inspect logs

In [ ]:
pf_result = load_csv(sim_name_pf)
emt_result = load_csv(sim_name_emt)

print("PF result:")
display(pf_result.head())

print("EMT result:")
display(emt_result.head())

## Plot selected EMT results

In [ ]:
time_col = emt_result.columns[0]
time = emt_result[time_col].to_numpy()

for prefix, ylabel, title in [
    ("P_elec", "P [W]", "Electrical active power"),
    ("Q_elec", "Q [var]", "Electrical reactive power"),
    ("VCO_Phase", "angle [rad]", "VCO output phase"),
]:
    cols = signal_columns(emt_result, prefix)
    if not cols:
        print(f"Skipping {prefix}: no columns found")
        continue

    plt.figure(figsize=(10, 4))
    for col in cols:
        plt.plot(time, emt_result[col], label=col)
    plt.xlabel("time [s]")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()

In [ ]:
voltage_signals = [
    ("Voltage_Node_1_PCC", "Voltage_Node_1_PCC"),
    ("Voltage_Node_2", "Voltage_Node_2"),
    ("Voltage_Node_3", "Voltage_Node_3"),
    ("Current_RLC", "Current_RLC"),
]

for title, pattern in voltage_signals:
    cols = signal_columns(emt_result, pattern)

    if not cols:
        print(f"No columns found for {title}.")
        continue

    plt.figure(figsize=(10, 4))
    for col in cols:
        plt.plot(time, emt_result[col], label=col)

    plt.xlabel("time [s]")
    plt.ylabel("voltage [V]")
    plt.title(f"{title}")
    plt.grid(True)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()